In [2]:
import subprocess

# Try git clone instead
result = subprocess.run([
    "git", "clone", 
    "https://github.com/FormAI-Dataset/FormAI-dataset.git"
], capture_output=True, text=True)

print(result.stdout)
print(result.stderr)

import os
print(os.listdir("."))


Cloning into 'FormAI-dataset'...
Updating files:  70% (7/10)
Updating files:  80% (8/10)
Updating files:  90% (9/10)
Updating files: 100% (10/10)
Updating files: 100% (10/10), done.

['formai.csv', 'FormAI-dataset', '.virtual_documents']


In [3]:
import os
import pandas as pd

print("Files in FormAI-dataset folder:")
print(os.listdir("FormAI-dataset"))

Files in FormAI-dataset folder:
['.DS_Store', '.git', 'README.md', 'FormAI-v2-DATASET.7z', 'FormAI-v2-classification.7z.001', 'FormAI-v2-classification.7z.002', 'FormAI_dataset_classification-V1.zip', 'FormAI-v2-DATASET_without_clones.7z', 'LICENSE', 'FormAI_dataset_C_samples-V1.zip', 'FormAI_dataset_human_readable-V1.csv']


In [4]:
import subprocess, os

# Unzip the classification file
subprocess.run(["unzip", "FormAI-dataset/FormAI_dataset_classification-V1.zip"], check=True)

print(os.listdir("."))

Archive:  FormAI-dataset/FormAI_dataset_classification-V1.zip
  inflating: FormAI_dataset.csv      
['formai.csv', 'FormAI-dataset', '.virtual_documents', 'FormAI_dataset.csv']


In [5]:
import pandas as pd

df = pd.read_csv("FormAI_dataset.csv")
print("Total rows:", len(df))
print("Columns:", list(df.columns))
print("\nFirst 2 rows:")
print(df.head(2))

Total rows: 246549
Columns: ['Filename', 'Vulnerability type', 'Source code', 'Function name', 'Line', 'Error type']

First 2 rows:
         Filename            Vulnerability type  \
0  FormAI_16037.c  NOT VULNERABLE up to bound k   
1  FormAI_11430.c  NOT VULNERABLE up to bound k   

                                         Source code Function name  Line  \
0  //FormAI DATASET v1.0 Category: Threading Libr...           NaN   NaN   
1  //FormAI DATASET v1.0 Category: Simple Web Ser...           NaN   NaN   

  Error type  
0        NaN  
1        NaN  


In [6]:
import pandas as pd

# Create binary label: 0 = not vulnerable, 1 = vulnerable
df['label'] = df['Vulnerability type'].apply(
    lambda x: 0 if 'NOT VULNERABLE' in str(x) else 1
)

print("Label distribution:")
print(df['label'].value_counts())
print("\nTotal:", len(df))
print("Missing code:", df['Source code'].isna().sum())

Label distribution:
label
1    201274
0     45275
Name: count, dtype: int64

Total: 246549
Missing code: 0


In [7]:
import re

def clean_code(code):
    code = re.sub(r'//.*', '', code)
    code = re.sub(r'/\*.*?\*/', '', code, flags=re.DOTALL)
    code = re.sub(r'\n\s*\n', '\n', code)
    return code.strip()

df['clean_code'] = df['Source code'].apply(clean_code)

# Balance: all safe + sample vulnerable (since vulnerable is majority here)
safe = df[df['label'] == 0]
vuln = df[df['label'] == 1].sample(n=len(safe)*3, random_state=42)
df_balanced = pd.concat([safe, vuln]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Safe:       {len(df_balanced[df_balanced['label']==0])}")
print(f"Vulnerable: {len(df_balanced[df_balanced['label']==1])}")
print(f"Total:      {len(df_balanced)}")

df_balanced[['clean_code', 'label']].to_csv('formai_balanced.csv', index=False)
print("Saved to formai_balanced.csv!")

Safe:       45275
Vulnerable: 135825
Total:      181100
Saved to formai_balanced.csv!


In [8]:
import os
files = ['formai_balanced.csv', 'diversevul_balanced.csv']
for f in files:
    if os.path.exists(f):
        size = os.path.getsize(f) / (1024*1024)
        print(f"✅ {f} — {size:.1f} MB")
    else:
        print(f"❌ {f} — MISSING")

✅ formai_balanced.csv — 363.0 MB
❌ diversevul_balanced.csv — MISSING
